In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings

# Machine Learning Models & Utils
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

# Configuration
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("muted")

print("Libraries Imported Successfully")

/usr/local/lib/python3.12/dist-packages/sqlalchemy/orm/query.py:195: SyntaxWarning: "is not" with 'tuple' literal. Did you mean "!="?
  if entities is not ():


Libraries Imported Successfully


In [2]:
# Initialize paths
train_path = ''
test_path = ''
submission_path = ''

# Search for dataset in /kaggle/input
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        full_path = os.path.join(dirname, filename)
        if 'train.csv' in filename:
            train_path = full_path
        elif 'test.csv' in filename:
            test_path = full_path
        elif 'sample_submission.csv' in filename:
            submission_path = full_path

# Load Data
train = pd.read_csv(train_path)
test = pd.read_csv(test_path)
submission = pd.read_csv(submission_path)

print(f"Train Shape: {train.shape}")
print(f"Test Shape: {test.shape}")


Train Shape: (700000, 26)
Test Shape: (300000, 25)


In [3]:
# 1. Separate ID (Save for submission)
test_ids = test['id']
train = train.drop('id', axis=1)
test = test.drop('id', axis=1)

# 2. Separate Target Variable
target = train['diagnosed_diabetes']
train = train.drop('diagnosed_diabetes', axis=1)

# 3. Combine Train/Test for Consistent Encoding
train_len = len(train)
combined = pd.concat([train, test], axis=0)

# 4. One-Hot Encoding (Convert categorical to dummy variables)
combined_encoded = pd.get_dummies(combined, drop_first=True)

# 5. Split back to Train/Test
X = combined_encoded.iloc[:train_len]
X_test = combined_encoded.iloc[train_len:]

# 6. Scaling (Standardization)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_test_scaled = scaler.transform(X_test)

print(f"Final Feature Count: {X.shape[1]}")

Final Feature Count: 36


In [4]:
# Split data (80% Train, 20% Validation)
# Stratify ensures the ratio of diabetes labels remains consistent
X_train, X_val, y_train, y_val = train_test_split(
    X_scaled, target, test_size=0.2, random_state=42, stratify=target
)

print(f"Train Set: {X_train.shape}")
print(f"Validation Set: {X_val.shape}")

Train Set: (560000, 36)
Validation Set: (140000, 36)


In [5]:
# Model 1 - XGBoost 

# Initialize XGBoost
xgb_model = XGBClassifier(
    n_estimators=2000,
    learning_rate=0.02,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='auc',
    random_state=42,
    n_jobs=-1,
    device='cpu',
    early_stopping_rounds=100  
)

# Train Model
print("Training XGBoost...")

xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_train, y_train), (X_val, y_val)],
    verbose=500  
)

# Validation Score
xgb_val_pred = xgb_model.predict_proba(X_val)[:, 1]
xgb_score = roc_auc_score(y_val, xgb_val_pred)
print(f"\nXGBoost Validation AUC: {xgb_score:.5f}")


Training XGBoost...
[0]	validation_0-auc:0.68569	validation_1-auc:0.68514
[500]	validation_0-auc:0.72812	validation_1-auc:0.71940
[1000]	validation_0-auc:0.74000	validation_1-auc:0.72378
[1500]	validation_0-auc:0.74842	validation_1-auc:0.72526
[1999]	validation_0-auc:0.75547	validation_1-auc:0.72584

XGBoost Validation AUC: 0.72584


In [6]:
from lightgbm import early_stopping, log_evaluation

# Initialize LightGBM
lgbm_model = LGBMClassifier(
    n_estimators=2000,
    learning_rate=0.02,
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.8,
    metric='auc',
    random_state=42,
    n_jobs=-1,
    verbosity=-1
)

# Train Model
print("Training LightGBM...")
lgbm_model.fit(
    X_train, y_train,
    eval_set=[(X_train, y_train), (X_val, y_val)],
    callbacks=[early_stopping(100), log_evaluation(500)]
)

# Validation Score
lgbm_val_pred = lgbm_model.predict_proba(X_val)[:, 1]
lgbm_score = roc_auc_score(y_val, lgbm_val_pred)
print(f"\nLightGBM Validation AUC: {lgbm_score:.5f}")

Training LightGBM...
Training until validation scores don't improve for 100 rounds
[500]	training's auc: 0.726938	valid_1's auc: 0.7222
[1000]	training's auc: 0.734246	valid_1's auc: 0.724482
[1500]	training's auc: 0.740006	valid_1's auc: 0.725422
[2000]	training's auc: 0.745291	valid_1's auc: 0.726033
Did not meet early stopping. Best iteration is:
[1981]	training's auc: 0.745102	valid_1's auc: 0.72604

LightGBM Validation AUC: 0.72604


In [7]:
# 1. Check Correlation (Should be high but < 1.0)
predictions = pd.DataFrame({'XGB': xgb_val_pred, 'LGBM': lgbm_val_pred})
print("Correlation between models:\n", predictions.corr())

# 2. Generate Predictions for Test Data
# Use best_iteration for XGBoost
xgb_test_pred = xgb_model.predict_proba(X_test_scaled, iteration_range=(0, xgb_model.best_iteration + 1))[:, 1]
lgbm_test_pred = lgbm_model.predict_proba(X_test_scaled)[:, 1]

# 3. Weighted Average (Adjust weights based on validation scores)
# If XGB is better, give it more weight (e.g., 0.6)
w_xgb = 0.5
w_lgbm = 0.5

final_pred = (xgb_test_pred * w_xgb) + (lgbm_test_pred * w_lgbm)

print(f"Ensemble Complete (Weights -> XGB: {w_xgb}, LGBM: {w_lgbm})")


Correlation between models:
            XGB      LGBM
XGB   1.000000  0.990693
LGBM  0.990693  1.000000
Ensemble Complete (Weights -> XGB: 0.5, LGBM: 0.5)


In [8]:
# Create DataFrame
submission_df = pd.DataFrame({
    'id': test_ids,
    'diagnosed_diabetes': final_pred
})

# Save to CSV
submission_df.to_csv('submission.csv', index=False)

print("'submission.csv' created successfully!")
display(submission_df.head())

'submission.csv' created successfully!


,id,diagnosed_diabetes
0,700000,0.493397
1,700001,0.665544
2,700002,0.763261
3,700003,0.404411
4,700004,0.926051
